# Experiment 5c: Qwen3.5-9B GGUF Q4_K_M through llama.cpp CUDA server

This notebook runs the existing typed v3 full LLM multi-agent TEC architecture on `Qwen/Qwen3.5-9B` via `lmstudio-community/Qwen3.5-9B-GGUF` and `Qwen3.5-9B-Q4_K_M.gguf`.

Experiment 5c replaces only the inference backend/model deployment path: local model adapter -> CUDA-enabled `llama.cpp` server with an OpenAI-compatible HTTP API. The typed v3 architecture, prompts, role protocol, tools, GoldRunner, five-task benchmark, expected sequences used for evaluation, and metrics remain unchanged.

Previous stronger-model attempts using AWQ and BNB deployment paths are treated as infrastructure failures and are excluded from model-quality comparisons.


## 1. Colab setup: project, dependencies, and llama.cpp build

Run this cell first in a GPU Colab runtime. It clones or updates `tec_agent_project`, installs only project/runtime dependencies needed for this GGUF path, clones `llama.cpp`, records its commit hash, and builds `llama-server` with CUDA enabled.

This branch does not install or use Transformers model loading, BitsAndBytes, AWQ, GPTQ, vLLM, SGLang, Ollama, or `llama-cpp-python` as the inference backend.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

PROJECT_ROOT = Path("/content/tec_agent_project")
REPO_URL = "https://github.com/ilyakosilov/tec_agent_project.git"
LLAMA_CPP_DIR = Path("/content/llama.cpp")
LLAMA_SERVER_BIN = LLAMA_CPP_DIR / "build" / "bin" / "llama-server"


def run(cmd, *, cwd=None, check=True):
    print("$", " ".join(str(x) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=cwd, check=check)


run(["nvidia-smi"], check=False)
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "cmake", "ninja-build", "build-essential", "git"])
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "huggingface_hub",
    "requests",
    "psutil",
    "pynvml",
    "pandas",
    "pyarrow",
    "numpy",
    "pydantic",
    "python-dateutil",
    "matplotlib",
    "xarray",
    "tqdm",
])

if PROJECT_ROOT.exists() and (PROJECT_ROOT / ".git").exists():
    run(["git", "-C", PROJECT_ROOT, "pull", "--ff-only"])
elif PROJECT_ROOT.exists():
    raise RuntimeError(f"{PROJECT_ROOT} exists but is not a git checkout.")
else:
    run(["git", "clone", REPO_URL, PROJECT_ROOT])

requirements_path = PROJECT_ROOT / "requirements.txt"
if requirements_path.exists():
    run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])

if (PROJECT_ROOT / "pyproject.toml").exists() or (PROJECT_ROOT / "setup.py").exists():
    run([sys.executable, "-m", "pip", "install", "-q", "-e", PROJECT_ROOT])
else:
    print("No pyproject.toml/setup.py found; project imports will use sys.path insertion.")

if LLAMA_CPP_DIR.exists():
    shutil.rmtree(LLAMA_CPP_DIR)
run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", LLAMA_CPP_DIR])
LLAMA_CPP_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=LLAMA_CPP_DIR,
    text=True,
).strip()
print("llama.cpp commit:", LLAMA_CPP_COMMIT)
run([
    "cmake",
    "-B",
    "build",
    "-G",
    "Ninja",
    "-DGGML_CUDA=ON",
    "-DCMAKE_BUILD_TYPE=Release",
], cwd=LLAMA_CPP_DIR)
run(["cmake", "--build", "build", "--config", "Release", "-j", "2", "--target", "llama-server"], cwd=LLAMA_CPP_DIR)
assert LLAMA_SERVER_BIN.exists(), f"llama-server binary not found: {LLAMA_SERVER_BIN}"
print("llama-server:", LLAMA_SERVER_BIN)


## 2. Project imports


In [ ]:
import json
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from tec_agents.agents.llm_multi_agent import initial_multi_agent_context
from tec_agents.agents.llm_multi_agent_typed import (
    LLMFullTypedMultiAgent,
    LLMTypedOrchestratorAgent,
)
from tec_agents.agents.llm_multi_agent_typed_prompts import ARCHITECTURE_NAME
from tec_agents.agents.llm_multi_agent_typed_protocol import TYPED_PROTOCOL_VERSION
from tec_agents.agents.llm_single_agent import infer_task_state
from tec_agents.data.datasets import get_dataset_summary, register_dataset
from tec_agents.eval.five_task_configs import (
    get_five_task_configs,
    build_five_task_eval_task,
    build_five_task_expected_sequence,
)
from tec_agents.eval.gold_runner import GoldRunner
from tec_agents.eval.metrics import MetricResult, compare_agent_to_gold
from tec_agents.eval.task_set import task_to_dict
from tec_agents.mcp.client import LocalMCPClient
from tec_agents.mcp.server import build_local_mcp_server

print("Project root:", PROJECT_ROOT)
print("Typed architecture:", ARCHITECTURE_NAME)
print("Typed protocol:", TYPED_PROTOCOL_VERSION)


## 3. CONFIG


In [ ]:
DATASET_REF = "default"
DATASET_FILENAME = "tec_regions_2024_01_01_to_2024_04_01_hourly.parquet"
PROCESSED_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / DATASET_FILENAME

BASE_MODEL_NAME = "Qwen/Qwen3.5-9B"
QUANTIZED_MODEL_REPO = "lmstudio-community/Qwen3.5-9B-GGUF"
GGUF_FILENAME = "Qwen3.5-9B-Q4_K_M.gguf"

MODEL_NAME = f"{QUANTIZED_MODEL_REPO}:{GGUF_FILENAME}"
MODEL_ABLATION_ID = "qwen35_9b_gguf_q4_k_m_llamacpp"
QUANTIZATION_FORMAT = "GGUF_Q4_K_M"
INFERENCE_BACKEND = "llama.cpp_cuda_server"

ARCHITECTURE_MODE = "qwen_multi_agent_typed_full_llm"
PROMPT_REVISION = "grounded_inputs_deliverables_single_block_v3"
EXPERIMENT_ID = "experiment_5c_qwen35_9b_gguf_q4_k_m_llamacpp"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "metrics"
OUTPUT_DIR = OUTPUT_ROOT / EXPERIMENT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_OUTPUT_PATH = OUTPUT_DIR / "qwen_multi_agent_typed_v3_qwen35_9b_gguf_q4_k_m_llamacpp_batch_colab.json"
PER_TASK_OUTPUT_TEMPLATE = "qwen_multi_agent_typed_v3_qwen35_9b_gguf_q4_k_m_llamacpp_{preset_id}_colab.json"
INFRASTRUCTURE_FAILURE_PATH = OUTPUT_DIR / "qwen35_9b_gguf_q4_k_m_llamacpp_infrastructure_smoke_failure.json"
PILOT_OUTPUT_DIR = OUTPUT_ROOT / f"{EXPERIMENT_ID}_pilot"
PILOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START = "2024-03-01"
END = "2024-04-01"

MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0
DO_SAMPLE = False

LLAMA_SERVER_HOST = "127.0.0.1"
LLAMA_SERVER_PORT = 8080
LLAMA_CONTEXT_SIZE = 8192
LLAMA_N_GPU_LAYERS = 99
LLAMA_N_GPU_LAYERS_FALLBACK = 50
LLAMA_SEED = 42
LLAMA_SERVER_LOG_PATH = Path("/content/llama_server_qwen35_9b_gguf.log")

MAX_ORCHESTRATION_STEPS = 12
MAX_ROLE_STEPS = 8
MAX_TOOL_CALLS_PER_ROLE = 8
MAX_PARSE_RETRIES = 2
MAX_TOOL_RETRIES = 2
STATE_FEEDBACK_MODE = "typed_state"

RUN_ALL_TASKS = True
SELECTED_PRESET = "hightec_midlat_europe"

SEMANTIC_SMOKE_PASSED = False
TYPED_XML_SMOKE_PASSED = False
ORCHESTRATOR_SMOKE_PASSED = False
PILOT_WORKFLOW_PASSED = False
ACTUAL_LLAMA_N_GPU_LAYERS = None
LLAMA_SERVER_CMD = []
GPU_NAME = None
GPU_MEMORY_TOTAL_MB = None
GPU_MEMORY_AFTER_MODEL_LOAD_MB = None
SERVER_START_FALLBACK_USED = False
SERVER_START_FALLBACK_REASON = None

TASK_CONFIGS = get_five_task_configs(dataset_ref=DATASET_REF)
if RUN_ALL_TASKS:
    SELECTED_TASK_CONFIGS = TASK_CONFIGS
else:
    SELECTED_TASK_CONFIGS = [c for c in TASK_CONFIGS if c["preset_id"] == SELECTED_PRESET]
assert SELECTED_TASK_CONFIGS, f"Unknown SELECTED_PRESET={SELECTED_PRESET!r}"

print("Architecture:", ARCHITECTURE_MODE)
print("Prompt revision:", PROMPT_REVISION)
print("Model ablation:", MODEL_ABLATION_ID)
print("Model:", MODEL_NAME)
print("Base model:", BASE_MODEL_NAME)
print("Quantization:", QUANTIZATION_FORMAT)
print("Backend:", INFERENCE_BACKEND)
print("Output dir:", OUTPUT_DIR)
print("Selected tasks:", [c["preset_id"] for c in SELECTED_TASK_CONFIGS])


## Planned test questions

This preview is for the human notebook reader only. The expected tool sequence is used later for evaluation and is not passed into the typed LLM agents.

In [ ]:
preview_rows = []
for config in SELECTED_TASK_CONFIGS:
    preview_rows.append({
        "preset_id": config["preset_id"],
        "task_type": config["task_type"],
        "query": config["query"],
        "expected_tool_sequence": " -> ".join(build_five_task_expected_sequence(config)),
    })
pd.DataFrame(preview_rows)

## 4. Dataset setup

In [ ]:
DATASET_PATH = PROCESSED_DATASET_PATH
assert DATASET_PATH.exists(), f"Missing dataset: {DATASET_PATH}"
register_dataset(dataset_ref=DATASET_REF, path=DATASET_PATH, file_format="parquet")
get_dataset_summary(DATASET_REF)

## 5. Download the single GGUF file

This cell downloads exactly `Qwen3.5-9B-Q4_K_M.gguf` from `lmstudio-community/Qwen3.5-9B-GGUF`. It does not download full BF16/FP16 weights or any previous failed deployment checkpoint.


In [ ]:
from huggingface_hub import hf_hub_download

GGUF_PATH = hf_hub_download(
    repo_id=QUANTIZED_MODEL_REPO,
    filename=GGUF_FILENAME,
    cache_dir="/content/hf_cache",
)
GGUF_PATH = Path(GGUF_PATH)
print("GGUF path:", GGUF_PATH)
print("GGUF size GB:", GGUF_PATH.stat().st_size / 1024**3)
assert GGUF_PATH.exists()
assert GGUF_PATH.stat().st_size > 4 * 1024**3


## 6. Shared helpers, metadata, and infrastructure failure recording


In [ ]:
def to_jsonable(value):
    if hasattr(value, "to_dict"):
        return to_jsonable(value.to_dict())
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    if hasattr(value, "item"):
        return value.item()
    return value


def metric_dict(metric_result):
    if metric_result is None:
        return {}
    return metric_result.metrics if isinstance(metric_result, MetricResult) else dict(metric_result)


def final_answer_preview(result, limit=360):
    text = getattr(result, "answer", "") or ""
    return text[:limit] + ("..." if len(text) > limit else "")


def build_verdict_checks(result, metric_result):
    metrics = metric_dict(metric_result)
    return {
        "agent_success": bool(result.success),
        "tool_sequence_match": metrics.get("tool_sequence_match"),
        "final_answer_present": bool(result.answer),
        "no_parse_errors": result.parse_error_count == 0,
        "no_stall": not result.stalled_loop_detected,
        "no_forbidden_tool_calls": result.forbidden_tool_call_count == 0,
        "no_premature_role_completion": result.premature_role_completion_count == 0,
        "no_empty_analysis_findings": result.empty_findings_done_count == 0,
    }


def overall_ok_from_checks(checks):
    required = [v for v in checks.values() if isinstance(v, bool)]
    return bool(required) and all(required)


def key_metric_summary(task_type, metrics):
    if task_type == "high_tec":
        return f"threshold_abs_error={metrics.get('threshold_abs_error')}; interval_count_error={metrics.get('interval_count_error')}"
    if task_type == "stable_intervals":
        return f"stable_interval_count_error={metrics.get('stable_interval_count_error')}"
    if task_type == "compare_regions":
        return f"mean_abs_error_avg={metrics.get('mean_abs_error_avg')}; pairwise_delta_count_match={metrics.get('pairwise_delta_count_match')}"
    if task_type == "report":
        return f"required_artifacts_present={metrics.get('required_artifacts_present')}; grounded={metrics.get('report_grounded_in_artifacts')}"
    return ""


def read_server_log_tail(n_lines=80):
    if not LLAMA_SERVER_LOG_PATH.exists():
        return ""
    lines = LLAMA_SERVER_LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(lines[-n_lines:])


def gpu_memory_snapshot():
    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,memory.used",
                "--format=csv,noheader,nounits",
            ],
            text=True,
            capture_output=True,
            check=True,
        )
        row = result.stdout.strip().splitlines()[0]
        name, total_mb, used_mb = [item.strip() for item in row.split(",")]
        return {
            "gpu_name": name,
            "memory_total_mb": int(total_mb),
            "memory_used_mb": int(used_mb),
        }
    except Exception as exc:
        return {"gpu_name": None, "memory_total_mb": None, "memory_used_mb": None, "error": repr(exc)}


def build_inference_metadata():
    return {
        "base_model_name": BASE_MODEL_NAME,
        "quantized_model_repo": QUANTIZED_MODEL_REPO,
        "gguf_filename": GGUF_FILENAME,
        "quantization_format": QUANTIZATION_FORMAT,
        "inference_backend": INFERENCE_BACKEND,
        "llama_cpp_commit": globals().get("LLAMA_CPP_COMMIT"),
        "llama_context_size": LLAMA_CONTEXT_SIZE,
        "llama_n_gpu_layers": globals().get("ACTUAL_LLAMA_N_GPU_LAYERS"),
        "llama_seed": LLAMA_SEED,
        "gpu_name": globals().get("GPU_NAME"),
        "gpu_memory_total_mb": globals().get("GPU_MEMORY_TOTAL_MB"),
        "gpu_memory_after_model_load_mb": globals().get("GPU_MEMORY_AFTER_MODEL_LOAD_MB"),
        "semantic_smoke_passed": bool(globals().get("SEMANTIC_SMOKE_PASSED")),
        "typed_xml_smoke_passed": bool(globals().get("TYPED_XML_SMOKE_PASSED")),
        "orchestrator_smoke_passed": bool(globals().get("ORCHESTRATOR_SMOKE_PASSED")),
        "pilot_workflow_passed": bool(globals().get("PILOT_WORKFLOW_PASSED")),
        "server_start_fallback_used": bool(globals().get("SERVER_START_FALLBACK_USED")),
        "server_start_fallback_reason": globals().get("SERVER_START_FALLBACK_REASON"),
        "server_command": list(globals().get("LLAMA_SERVER_CMD") or []),
        "gguf_path": str(globals().get("GGUF_PATH", "")),
    }


def save_infrastructure_failure(failed_smoke_gate, *, raw_smoke_output=None, error=None, server_cmd=None):
    payload = {
        "verdict": "Infrastructure validation failed. This run must not be included in model-quality or architecture comparison.",
        "failed_smoke_gate": failed_smoke_gate,
        "model_name": MODEL_NAME,
        "base_model_name": BASE_MODEL_NAME,
        "quantized_model_repo": QUANTIZED_MODEL_REPO,
        "gguf_filename": GGUF_FILENAME,
        "inference_backend": INFERENCE_BACKEND,
        "llama_cpp_commit": globals().get("LLAMA_CPP_COMMIT"),
        "server_command": list(server_cmd or globals().get("LLAMA_SERVER_CMD") or []),
        "gpu_memory": gpu_memory_snapshot(),
        "raw_smoke_output": raw_smoke_output,
        "error": repr(error) if error is not None else None,
        "server_log_tail": read_server_log_tail(),
        "inference_metadata": build_inference_metadata(),
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with INFRASTRUCTURE_FAILURE_PATH.open("w", encoding="utf-8") as f:
        json.dump(to_jsonable(payload), f, ensure_ascii=False, indent=2)
    print(payload["verdict"])
    print("Saved infrastructure failure:", INFRASTRUCTURE_FAILURE_PATH)


## 7. Start CUDA-enabled llama.cpp server

The server is launched as a subprocess and exposes an OpenAI-compatible endpoint at `/v1/chat/completions`. If `-ngl 99` cannot fit, the notebook makes one explicit fallback attempt with `-ngl 50`, records it in metadata, and still requires all smoke-gates before benchmark.


In [ ]:
import psutil


def stop_previous_llama_server():
    global LLAMA_SERVER_PROCESS
    proc = globals().get("LLAMA_SERVER_PROCESS")
    if proc is not None and proc.poll() is None:
        print("Stopping previous llama-server PID:", proc.pid)
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
    for candidate in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            cmdline = candidate.info.get("cmdline") or []
            cmdline_text = " ".join(cmdline)
            if "llama-server" in cmdline_text and str(LLAMA_SERVER_PORT) in cmdline_text:
                print("Stopping existing llama-server process:", candidate.info["pid"])
                candidate.terminate()
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass


def wait_for_llama_server(endpoint, attempts=60, delay_sec=2):
    last_error = None
    for _ in range(attempts):
        try:
            response = requests.get(f"{endpoint}/health", timeout=5)
            if response.ok:
                return True
        except Exception as exc:
            last_error = exc
        try:
            response = requests.get(f"{endpoint}/v1/models", timeout=5)
            if response.ok:
                return True
        except Exception as exc:
            last_error = exc
        if globals().get("LLAMA_SERVER_PROCESS") is not None and LLAMA_SERVER_PROCESS.poll() is not None:
            break
        time.sleep(delay_sec)
    if last_error:
        print("Last health-check error:", repr(last_error))
    return False


def start_llama_server(n_gpu_layers):
    global LLAMA_SERVER_PROCESS, LLAMA_SERVER_CMD
    stop_previous_llama_server()
    server_cmd = [
        str(LLAMA_SERVER_BIN),
        "-m", str(GGUF_PATH),
        "--host", LLAMA_SERVER_HOST,
        "--port", str(LLAMA_SERVER_PORT),
        "-c", str(LLAMA_CONTEXT_SIZE),
        "-ngl", str(n_gpu_layers),
        "--seed", str(LLAMA_SEED),
        "--jinja",
    ]
    LLAMA_SERVER_CMD = server_cmd
    print("Starting llama-server:", " ".join(server_cmd))
    log_file = LLAMA_SERVER_LOG_PATH.open("w", encoding="utf-8")
    LLAMA_SERVER_PROCESS = subprocess.Popen(
        server_cmd,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )
    endpoint = f"http://{LLAMA_SERVER_HOST}:{LLAMA_SERVER_PORT}"
    if not wait_for_llama_server(endpoint):
        print(read_server_log_tail())
        raise RuntimeError(f"llama-server failed to start with n_gpu_layers={n_gpu_layers}")
    return endpoint


try:
    ENDPOINT = start_llama_server(LLAMA_N_GPU_LAYERS)
    ACTUAL_LLAMA_N_GPU_LAYERS = LLAMA_N_GPU_LAYERS
except Exception as exc:
    SERVER_START_FALLBACK_USED = True
    SERVER_START_FALLBACK_REASON = repr(exc)
    print("Initial llama-server start failed; attempting explicit n_gpu_layers fallback.")
    print("Reason:", SERVER_START_FALLBACK_REASON)
    try:
        ENDPOINT = start_llama_server(LLAMA_N_GPU_LAYERS_FALLBACK)
        ACTUAL_LLAMA_N_GPU_LAYERS = LLAMA_N_GPU_LAYERS_FALLBACK
    except Exception as fallback_exc:
        save_infrastructure_failure("llama_server_start", error=fallback_exc, server_cmd=LLAMA_SERVER_CMD)
        raise

memory_after_load = gpu_memory_snapshot()
GPU_NAME = memory_after_load.get("gpu_name")
GPU_MEMORY_TOTAL_MB = memory_after_load.get("memory_total_mb")
GPU_MEMORY_AFTER_MODEL_LOAD_MB = memory_after_load.get("memory_used_mb")
print("llama.cpp endpoint:", ENDPOINT)
print("model path:", GGUF_PATH)
print("server PID:", LLAMA_SERVER_PROCESS.pid)
print("n_gpu_layers:", ACTUAL_LLAMA_N_GPU_LAYERS)
print("GPU memory after model load:", memory_after_load)


## 8. llama.cpp HTTP chat adapter

The typed runner only needs an object named `model` with a `.generate(messages, ...) -> str` method. This adapter calls the local `llama-server` OpenAI-compatible HTTP endpoint and does not modify prompts or structured XML blocks.


In [ ]:
class Qwen35GGUFLlamaCppChatModel:
    def __init__(self, endpoint: str, model_name: str, seed: int = 42, timeout_sec: int = 600):
        self.endpoint = endpoint.rstrip("/")
        self.model_name = model_name
        self.seed = seed
        self.timeout_sec = timeout_sec

    def generate(
        self,
        messages,
        max_new_tokens=512,
        temperature=0.0,
        do_sample=False,
        stop=None,
    ) -> str:
        payload = {
            "model": self.model_name,
            "messages": messages,
            "max_tokens": int(max_new_tokens),
            "temperature": float(temperature if do_sample else 0.0),
            "seed": self.seed,
            "stream": False,
        }
        response = requests.post(
            f"{self.endpoint}/v1/chat/completions",
            json=payload,
            timeout=self.timeout_sec,
        )
        if not response.ok:
            raise RuntimeError(
                f"llama.cpp request failed: {response.status_code}\n{response.text[:2000]}"
            )
        data = response.json()
        try:
            text = data["choices"][0]["message"]["content"]
        except Exception as exc:
            raise RuntimeError(f"Unexpected llama.cpp response schema: {data}") from exc
        if stop:
            cut_at = None
            for item in stop:
                idx = text.find(item)
                if idx >= 0 and (cut_at is None or idx < cut_at):
                    cut_at = idx
            if cut_at is not None:
                text = text[:cut_at]
        return text


model = Qwen35GGUFLlamaCppChatModel(
    endpoint=f"http://{LLAMA_SERVER_HOST}:{LLAMA_SERVER_PORT}",
    model_name=MODEL_NAME,
    seed=LLAMA_SEED,
)
print("Created llama.cpp HTTP adapter for:", MODEL_NAME)


## 9. Mandatory infrastructure smoke-gates

The full five-task benchmark must not run unless all smoke-gates pass: semantic generation, XML block reproduction, and a first real Orchestrator prompt parsed by the existing typed protocol.


In [ ]:
try:
    semantic_messages = [
        {"role": "system", "content": "You are a precise assistant. Follow the user's instruction exactly."},
        {"role": "user", "content": "Reply with exactly one word: ready"},
    ]
    semantic_output = model.generate(semantic_messages, max_new_tokens=16, temperature=0.0, do_sample=False)
    print("Semantic smoke output:", repr(semantic_output))
    assert semantic_output.strip().lower() == "ready", (
        "Semantic smoke test failed. The GGUF/llama.cpp inference path is not valid "
        f"for agent benchmarking. Output: {semantic_output!r}"
    )
    SEMANTIC_SMOKE_PASSED = True
except Exception as exc:
    save_infrastructure_failure("semantic_smoke", raw_smoke_output=globals().get("semantic_output"), error=exc)
    raise

try:
    xml_messages = [
        {"role": "system", "content": "Return exactly one XML block and nothing else."},
        {
            "role": "user",
            "content": (
                "Return exactly this text:\n"
                "<role_action>\n"
                "{\"role\":\"DataAgent\",\"deliverables_to_produce\":[\"series_id\"]}\n"
                "</role_action>"
            ),
        },
    ]
    xml_output = model.generate(xml_messages, max_new_tokens=96, temperature=0.0, do_sample=False)
    print("XML smoke output:", repr(xml_output))
    for fragment in ["<role_action>", "</role_action>", "DataAgent", "series_id"]:
        assert fragment in xml_output, f"Missing XML smoke fragment: {fragment!r}"
    TYPED_XML_SMOKE_PASSED = True
except Exception as exc:
    save_infrastructure_failure("typed_xml_smoke", raw_smoke_output=globals().get("xml_output"), error=exc)
    raise

try:
    hightec_config = next(c for c in TASK_CONFIGS if c["preset_id"] == "hightec_midlat_europe")
    hightec_query = hightec_config["query"]
    parsed_task = infer_task_state(hightec_query)
    context = initial_multi_agent_context(user_query=hightec_query, parsed_task=parsed_task)
    orchestrator = LLMTypedOrchestratorAgent(model, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS)
    role_action, orchestrator_diagnostics = orchestrator.decide(
        user_query=hightec_query,
        context=context,
        max_parse_retries=0,
    )
    print("Raw first orchestrator output:")
    print((orchestrator_diagnostics.get("raw_model_outputs") or [""])[-1])
    print("Cleaned first orchestrator output:")
    print((orchestrator_diagnostics.get("cleaned_model_outputs") or [""])[-1])
    print("Parsed role action:", role_action)
    print("Validation diagnostics:", orchestrator_diagnostics)
    assert role_action is not None, "Orchestrator did not produce a parseable RoleAction."
    assert role_action.action in {"call_role", "finish"}
    if role_action.action == "call_role":
        assert role_action.role in {"data_agent", "math_agent", "analysis_agent", "report_agent"}
        assert role_action.assignment is not None
    assert orchestrator_diagnostics.get("parse_error_count", 0) == 0
    ORCHESTRATOR_SMOKE_PASSED = True
except Exception as exc:
    save_infrastructure_failure(
        "orchestrator_smoke",
        raw_smoke_output=(globals().get("orchestrator_diagnostics") or {}).get("raw_model_outputs"),
        error=exc,
    )
    raise

print("Infrastructure smoke-gates passed.")


## 10. Pilot workflow gate

Before running the full five-task batch, run only `hightec_midlat_europe` in a separate pilot output directory. The pilot must execute at least one tool call. If it does not, the full benchmark stops.


In [ ]:
try:
    pilot_config = next(c for c in TASK_CONFIGS if c["preset_id"] == "hightec_midlat_europe")
    pilot_query = pilot_config["query"]
    print("Pilot query:", pilot_query)
    pilot_server = build_local_mcp_server(run_id="pilot_qwen35_9b_gguf_q4_k_m_llamacpp_hightec_midlat_europe_colab")
    pilot_client = LocalMCPClient(pilot_server)
    pilot_agent = LLMFullTypedMultiAgent(
        model=model,
        client=pilot_client,
        max_orchestration_steps=MAX_ORCHESTRATION_STEPS,
        max_role_steps=MAX_ROLE_STEPS,
        max_tool_calls_per_role=MAX_TOOL_CALLS_PER_ROLE,
        max_parse_retries=MAX_PARSE_RETRIES,
        max_tool_retries=MAX_TOOL_RETRIES,
        temperature=TEMPERATURE,
        state_feedback_mode=STATE_FEEDBACK_MODE,
    )
    pilot_result = pilot_agent.run(pilot_query)
    pilot_payload = {
        "experiment_id": f"{EXPERIMENT_ID}_pilot",
        "selected_preset": pilot_config["preset_id"],
        "task_config": pilot_config,
        "query": pilot_query,
        "architecture": ARCHITECTURE_MODE,
        "typed_protocol_version": TYPED_PROTOCOL_VERSION,
        "prompt_revision": PROMPT_REVISION,
        "model_name": MODEL_NAME,
        "model_ablation_id": MODEL_ABLATION_ID,
        "inference_metadata": build_inference_metadata(),
        "agent_result": pilot_result.to_dict(),
        "actual_tool_sequence": pilot_result.actual_tool_sequence,
        "tool_call_count": len(pilot_result.actual_tool_sequence),
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    pilot_path = PILOT_OUTPUT_DIR / "qwen_multi_agent_typed_v3_qwen35_9b_gguf_q4_k_m_llamacpp_hightec_midlat_europe_pilot_colab.json"
    with pilot_path.open("w", encoding="utf-8") as f:
        json.dump(to_jsonable(pilot_payload), f, ensure_ascii=False, indent=2)
    print("Saved pilot:", pilot_path)
    print("Pilot tool sequence:", pilot_result.actual_tool_sequence)
    assert len(pilot_result.actual_tool_sequence) >= 1, "Pilot workflow produced zero tool calls."
    PILOT_WORKFLOW_PASSED = True
except Exception as exc:
    save_infrastructure_failure(
        "pilot_workflow",
        raw_smoke_output=(globals().get("pilot_result").to_dict() if globals().get("pilot_result") is not None else None),
        error=exc,
    )
    raise

print("Pilot workflow passed. Full five-task batch may run.")


## 11. Batch loop

This is the same typed v3 five-task workflow: `result = agent.run(query)`, then GoldRunner and metrics run only after the agent finishes. Expected tool sequences are evaluation metadata only and are not passed into prompts.


In [ ]:
all_results = []
gold_runner = GoldRunner()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert SEMANTIC_SMOKE_PASSED and TYPED_XML_SMOKE_PASSED and ORCHESTRATOR_SMOKE_PASSED and PILOT_WORKFLOW_PASSED

for task_config in SELECTED_TASK_CONFIGS:
    preset_id = task_config["preset_id"]
    query = task_config["query"]
    expected_sequence = build_five_task_expected_sequence(task_config)
    eval_task = build_five_task_eval_task(task_config)

    print()
    print("===", preset_id, "===")
    server = build_local_mcp_server(run_id=f"qwen_multi_agent_typed_v3_qwen35_9b_gguf_q4_k_m_llamacpp_{preset_id}_colab")
    client = LocalMCPClient(server)
    agent = LLMFullTypedMultiAgent(
        model=model,
        client=client,
        max_orchestration_steps=MAX_ORCHESTRATION_STEPS,
        max_role_steps=MAX_ROLE_STEPS,
        max_tool_calls_per_role=MAX_TOOL_CALLS_PER_ROLE,
        max_parse_retries=MAX_PARSE_RETRIES,
        max_tool_retries=MAX_TOOL_RETRIES,
        temperature=TEMPERATURE,
        state_feedback_mode=STATE_FEEDBACK_MODE,
    )

    result = agent.run(query)
    result_dict = result.to_dict()

    gold = gold_runner.run(eval_task)
    metric_result = None
    gold_status = gold.status
    gold_error = gold.error
    gold_result = gold.result
    if gold.status == "success" and gold.result is not None:
        agent_metric_payload = dict(result.tool_results)
        agent_metric_payload.update({
            "parse_error_count": result.parse_error_count,
            "invalid_json_count": result.invalid_json_count,
            "invalid_role_protocol_count": result.invalid_role_protocol_count,
            "forbidden_tool_call_count": result.forbidden_tool_call_count,
            "repeated_tool_call_count": result.repeated_tool_call_count,
            "multiple_protocol_blocks_in_single_output_count": result.multiple_protocol_blocks_in_single_output_count,
            "invalid_artifact_handle_count": result.invalid_artifact_handle_count,
            "stalled_loop_detected": result.stalled_loop_detected,
        })
        metric_result = compare_agent_to_gold(
            task_id=eval_task.task_id,
            task_type=eval_task.task_type,
            agent_result=agent_metric_payload,
            gold_result=gold.result,
            agent_trace=result.trace,
            task=task_to_dict(eval_task),
            parsed_task=result.parsed_task,
            orchestration_steps=result.orchestration_steps,
        )

    metrics = metric_dict(metric_result)
    verdict_checks = build_verdict_checks(result, metric_result)
    overall_ok = overall_ok_from_checks(verdict_checks)

    record = {
        "selected_preset": preset_id,
        "task_config": task_config,
        "query": query,
        "expected_tool_sequence": expected_sequence,
        "actual_tool_sequence": result.actual_tool_sequence,
        "architecture": ARCHITECTURE_MODE,
        "model_name": MODEL_NAME,
        "model_ablation_id": MODEL_ABLATION_ID,
        "base_model_name": BASE_MODEL_NAME,
        "quantized_model_repo": QUANTIZED_MODEL_REPO,
        "gguf_filename": GGUF_FILENAME,
        "quantization_format": QUANTIZATION_FORMAT,
        "inference_backend": INFERENCE_BACKEND,
        "inference_metadata": build_inference_metadata(),
        "typed_protocol_version": TYPED_PROTOCOL_VERSION,
        "prompt_revision": PROMPT_REVISION,
        "agent_result": result_dict,
        "gold_status": gold_status,
        "gold_error": gold_error,
        "gold_result": gold_result,
        "metrics": metrics,
        "verdict_checks": verdict_checks,
        "overall_ok": overall_ok,
        "success": bool(result.success),
        "final_answer_preview": final_answer_preview(result),
        "role_agent_order": result.role_agent_order,
        "orchestrator_decisions": result.orchestrator_decisions,
        "role_actions": result.role_actions,
        "role_assignments": result.role_assignments,
        "role_outputs": result.role_outputs,
        "tool_observations": result.tool_observations,
        "available_artifacts_snapshots": result.available_artifacts_snapshots,
        "invalid_assignment_count": result.invalid_assignment_count,
        "invalid_role_action_count": result.invalid_role_action_count,
        "invalid_role_response_count": result.invalid_role_response_count,
        "invalid_final_answer_count": result.invalid_final_answer_count,
        "invalid_tool_name_count": result.invalid_tool_name_count,
        "forbidden_tool_call_count": result.forbidden_tool_call_count,
        "multiple_protocol_blocks_in_single_output_count": result.multiple_protocol_blocks_in_single_output_count,
        "invalid_artifact_handle_count": result.invalid_artifact_handle_count,
        "premature_role_completion_count": result.premature_role_completion_count,
        "empty_findings_done_count": result.empty_findings_done_count,
        "repeated_equivalent_role_assignment_count": result.repeated_equivalent_role_assignment_count,
        "tool_error_count": result.tool_error_count,
        "tool_schema_validation_error_count": result.tool_schema_validation_error_count,
        "invalid_role_protocol_count": result.invalid_role_protocol_count,
        "parse_error_count": result.parse_error_count,
        "invalid_json_count": result.invalid_json_count,
        "repeated_tool_call_count": result.repeated_tool_call_count,
        "stalled_loop_detected": result.stalled_loop_detected,
        "repair_attempt_count": result.repair_attempt_count,
        "retry_count": result.retry_count,
        "recovery_attempt_count": result.recovery_attempt_count,
        "recovery_success_count": result.recovery_success_count,
        "recovery_failure_count": result.recovery_failure_count,
        "key_metric_summary": key_metric_summary(task_config["task_type"], metrics),
        "missing_agent_terminal_artifact": (metrics or {}).get("missing_agent_terminal_artifact"),
    }

    per_task_path = OUTPUT_DIR / PER_TASK_OUTPUT_TEMPLATE.format(preset_id=preset_id)
    with per_task_path.open("w", encoding="utf-8") as f:
        json.dump(to_jsonable(record), f, ensure_ascii=False, indent=2)
    print("Saved:", per_task_path)
    print("success:", record["success"], "overall_ok:", overall_ok, "tools:", result.actual_tool_sequence)
    all_results.append(record)


## 12. Summary table and save batch


In [ ]:
def rate(values):
    values = [v for v in values if isinstance(v, bool)]
    return None if not values else sum(values) / len(values)

summary = {
    "n_tasks": len(all_results),
    "n_success": sum(1 for r in all_results if r["success"]),
    "n_failed": sum(1 for r in all_results if not r["success"]),
    "success_rate": rate([r["success"] for r in all_results]),
    "n_overall_ok": sum(1 for r in all_results if r["overall_ok"]),
    "overall_ok_rate": rate([r["overall_ok"] for r in all_results]),
    "agent_success_rate": rate([r["success"] for r in all_results]),
    "tool_sequence_match_rate": rate([(r.get("metrics") or {}).get("tool_sequence_match") for r in all_results]),
    "role_agent_order_match_rate": rate([(r.get("metrics") or {}).get("role_agent_order_match") for r in all_results]),
    "artifact_flow_valid_rate": rate([(r.get("metrics") or {}).get("artifact_flow_valid") for r in all_results]),
    "required_role_agents_called_rate": rate([(r.get("metrics") or {}).get("required_role_agents_called") for r in all_results]),
    "avg_tool_call_count": sum(len(r["actual_tool_sequence"]) for r in all_results) / max(1, len(all_results)),
    "avg_tool_error_count": sum(r["tool_error_count"] for r in all_results) / max(1, len(all_results)),
    "avg_parse_error_count": sum(r["parse_error_count"] for r in all_results) / max(1, len(all_results)),
    "avg_invalid_assignment_count": sum(r["invalid_assignment_count"] for r in all_results) / max(1, len(all_results)),
    "avg_invalid_role_response_count": sum(r["invalid_role_response_count"] for r in all_results) / max(1, len(all_results)),
    "avg_forbidden_tool_call_count": sum(r["forbidden_tool_call_count"] for r in all_results) / max(1, len(all_results)),
    "avg_multiple_protocol_blocks_in_single_output_count": sum(r["multiple_protocol_blocks_in_single_output_count"] for r in all_results) / max(1, len(all_results)),
    "avg_invalid_artifact_handle_count": sum(r["invalid_artifact_handle_count"] for r in all_results) / max(1, len(all_results)),
    "avg_premature_role_completion_count": sum(r["premature_role_completion_count"] for r in all_results) / max(1, len(all_results)),
    "avg_empty_findings_done_count": sum(r["empty_findings_done_count"] for r in all_results) / max(1, len(all_results)),
    "avg_repeated_equivalent_role_assignment_count": sum(r["repeated_equivalent_role_assignment_count"] for r in all_results) / max(1, len(all_results)),
    "avg_tool_schema_validation_error_count": sum(r["tool_schema_validation_error_count"] for r in all_results) / max(1, len(all_results)),
    "stalled_loop_rate": rate([r["stalled_loop_detected"] for r in all_results]),
    "legacy_report_tool_used_rate": rate([("tec_" + "build_report") in r["actual_tool_sequence"] for r in all_results]),
}

payload = {
    "experiment_id": EXPERIMENT_ID,
    "architecture": ARCHITECTURE_MODE,
    "model_name": MODEL_NAME,
    "model_ablation_id": MODEL_ABLATION_ID,
    "base_model_name": BASE_MODEL_NAME,
    "quantized_model_repo": QUANTIZED_MODEL_REPO,
    "gguf_filename": GGUF_FILENAME,
    "quantization_format": QUANTIZATION_FORMAT,
    "inference_backend": INFERENCE_BACKEND,
    "inference_metadata": build_inference_metadata(),
    "dataset_ref": DATASET_REF,
    "dataset_path": str(DATASET_PATH),
    "model_config": {
        "model_name": MODEL_NAME,
        "base_model_name": BASE_MODEL_NAME,
        "quantized_model_repo": QUANTIZED_MODEL_REPO,
        "gguf_filename": GGUF_FILENAME,
        "quantization_format": QUANTIZATION_FORMAT,
        "inference_backend": INFERENCE_BACKEND,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "do_sample": DO_SAMPLE,
    },
    "agent_config": {
        "max_orchestration_steps": MAX_ORCHESTRATION_STEPS,
        "max_role_steps": MAX_ROLE_STEPS,
        "max_tool_calls_per_role": MAX_TOOL_CALLS_PER_ROLE,
        "max_parse_retries": MAX_PARSE_RETRIES,
        "max_tool_retries": MAX_TOOL_RETRIES,
        "state_feedback_mode": STATE_FEEDBACK_MODE,
    },
    "typed_protocol_version": TYPED_PROTOCOL_VERSION,
    "prompt_revision": PROMPT_REVISION,
    "summary": summary,
    "results": all_results,
    "created_at": datetime.now(timezone.utc).isoformat(),
}

with BATCH_OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(to_jsonable(payload), f, ensure_ascii=False, indent=2)

summary_rows = []
for r in all_results:
    summary_rows.append({
        "preset_id": r["selected_preset"],
        "task_type": r["task_config"]["task_type"],
        "agent_success": r["success"],
        "overall_ok": r["overall_ok"],
        "tool_sequence_match": (r.get("metrics") or {}).get("tool_sequence_match"),
        "actual_tool_sequence": " -> ".join(r["actual_tool_sequence"]),
        "role_agent_order": " -> ".join(r["role_agent_order"]),
        "tool_call_count": len(r["actual_tool_sequence"]),
        "tool_error_count": r["tool_error_count"],
        "final_answer_present": bool((r["agent_result"] or {}).get("answer")),
        "role_agent_order_match": (r.get("metrics") or {}).get("role_agent_order_match"),
        "required_role_agents_called": (r.get("metrics") or {}).get("required_role_agents_called"),
        "artifact_flow_valid": (r.get("metrics") or {}).get("artifact_flow_valid"),
        "invalid_artifact_handle_count": r["invalid_artifact_handle_count"],
        "multiple_protocol_blocks_in_single_output_count": r["multiple_protocol_blocks_in_single_output_count"],
        "premature_role_completion_count": r["premature_role_completion_count"],
        "empty_findings_done_count": r["empty_findings_done_count"],
        "repeated_equivalent_role_assignment_count": r["repeated_equivalent_role_assignment_count"],
        "tool_schema_validation_error_count": r["tool_schema_validation_error_count"],
        "parse_error_count": r["parse_error_count"],
        "repeated_tool_call_count": r["repeated_tool_call_count"],
        "forbidden_tool_call_count": r["forbidden_tool_call_count"],
        "stalled_loop_detected": r["stalled_loop_detected"],
        "missing_agent_terminal_artifact": r.get("missing_agent_terminal_artifact"),
        "key_metric_summary": r["key_metric_summary"],
        "answer_preview": r["final_answer_preview"],
    })
print("Saved batch:", BATCH_OUTPUT_PATH)
pd.DataFrame(summary_rows)


## 13. Optional comparison

This cell only reads existing JSON. It compares the same typed v3 architecture on the previous `Qwen3.5-4B` run and this GGUF/llama.cpp run when both outputs are present. Historical infrastructure-failed stronger-model deployment attempts are not included as model-quality results.


In [ ]:
def load_optional(path):
    if not path.exists():
        print("Missing:", path)
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def records_by_preset(batch):
    if not batch:
        return {}
    return {r.get("selected_preset"): r for r in batch.get("results", [])}


def seq_text(record):
    if not record:
        return None
    return " -> ".join(record.get("actual_tool_sequence") or [])


def summary_row(label, batch):
    if not batch:
        return None
    summary = batch.get("summary") or {}
    return {
        "model": label,
        "success_rate": summary.get("success_rate"),
        "overall_ok_rate": summary.get("overall_ok_rate"),
        "tool_sequence_match_rate": summary.get("tool_sequence_match_rate"),
        "role_order_match_rate": summary.get("role_agent_order_match_rate"),
        "artifact_flow_valid_rate": summary.get("artifact_flow_valid_rate"),
        "avg_tool_call_count": summary.get("avg_tool_call_count"),
        "avg_tool_error_count": summary.get("avg_tool_error_count"),
        "avg_invalid_artifact_handle_count": summary.get("avg_invalid_artifact_handle_count"),
        "stalled_loop_rate": summary.get("stalled_loop_rate"),
    }


comparison_sources = {
    "typed_v3_qwen35_4b": load_optional(OUTPUT_ROOT / "experiment_4" / "qwen_multi_agent_typed_v3_batch_colab.json"),
    "typed_v3_qwen35_9b_gguf_llamacpp": load_optional(BATCH_OUTPUT_PATH),
    "rule_multi": load_optional(OUTPUT_ROOT / "multi_agent_rule_based_five_task_batch.json"),
}

four_b = records_by_preset(comparison_sources["typed_v3_qwen35_4b"])
gguf = records_by_preset(comparison_sources["typed_v3_qwen35_9b_gguf_llamacpp"])
comparison_rows = []
for config in SELECTED_TASK_CONFIGS:
    preset_id = config["preset_id"]
    r4 = four_b.get(preset_id)
    rg = gguf.get(preset_id)
    comparison_rows.append({
        "preset": preset_id,
        "task_type": config["task_type"],
        "Qwen3.5-4B typed v3 success": None if r4 is None else r4.get("success"),
        "Qwen3.5-9B GGUF typed v3 success": None if rg is None else rg.get("success"),
        "4B tool sequence": seq_text(r4),
        "GGUF tool sequence": seq_text(rg),
        "4B stalled": None if r4 is None else r4.get("stalled_loop_detected"),
        "GGUF stalled": None if rg is None else rg.get("stalled_loop_detected"),
        "4B invalid handles": None if r4 is None else r4.get("invalid_artifact_handle_count"),
        "GGUF invalid handles": None if rg is None else rg.get("invalid_artifact_handle_count"),
    })

aggregate_rows = [
    row for row in [
        summary_row("Qwen3.5-4B typed v3", comparison_sources["typed_v3_qwen35_4b"]),
        summary_row("Qwen3.5-9B GGUF Q4_K_M llama.cpp typed v3", comparison_sources["typed_v3_qwen35_9b_gguf_llamacpp"]),
    ] if row is not None
]

print("Model-ablation per-task comparison")
display(pd.DataFrame(comparison_rows))
print("Model-ablation aggregate comparison")
display(pd.DataFrame(aggregate_rows))
